# 14-01 从状态转移训练到多步预测

对应第 14 章。按顺序运行全部单元格，CPU 即可；依赖 NumPy、Matplotlib、PyTorch，本地使用项目的 .venv-colab 环境。图中文字使用英文以避免缺少中文字体，图后用中文逐项解释。

这里训练小型神经网络，数据来自一维教学接触规则：推杆向右靠近物体，距离小于规定间隙时推动它；向左离开不会拉动物体。这不是 PushT，也不模拟惯性、摩擦或旋转。先看真实交互，再训练，最后分别检查单步和连续预测。

## 1. 先生成完整轨迹，再划分训练与测试

位置单位为米，每个动作表示一个周期内推杆的位移。每条轨迹有 24 个动作、25 个状态。先划分轨迹编号，再抽出相邻状态，保证测试集没有混入训练轨迹的下一帧。第一幅图中两条线分别是推杆与物体；不要期待它们永远同时移动。

In [ ]:
import numpy as np  # 用数组保存轨迹与动作
import matplotlib.pyplot as plt  # 绘制位置曲线
import torch  # 使用神经网络训练与梯度计算
from torch import nn  # 使用线性层和激活函数
torch.set_num_threads(1)  # 限制线程数以便小模型在普通处理器上稳定运行
torch.manual_seed(14)  # 固定模型初始化与训练采样随机性
rng = np.random.default_rng(14)  # 固定教学数据的随机数种子
gap = 0.025  # 设定接触时推杆中心与物体中心的最小间距
def transition(state, action):  # 定义只用于生成答案的教学接触规则
    p = state[0] + float(action)  # 根据实际位移更新推杆位置
    b = max(state[1], p + gap)  # 接触时推动物体向右移动而不产生吸附
    return np.array([p, b], dtype=np.float32)  # 返回下一时刻两个位置
episodes = []  # 保存完整轨迹以便按轨迹划分数据
for episode in range(100):  # 生成一百条不同起点与动作的轨迹
    p = rng.uniform(0.05, 0.3)  # 随机选择推杆初始位置
    states = [np.array([p, p + rng.uniform(0.04, 0.16)], dtype=np.float32)]  # 将物体放在推杆右侧
    actions = rng.uniform(-0.015, 0.025, size=24).astype(np.float32)  # 生成靠近与离开的位移命令
    for action in actions:  # 依次执行当前轨迹的全部动作
        states.append(transition(states[-1], action))  # 记录动作之后的真实下一状态
    episodes.append((np.stack(states), actions))  # 将状态序列与区间动作一起保存
fig, ax = plt.subplots(figsize=(7, 3))  # 为一条真实交互创建简洁画布
ax.plot(episodes[0][0][:, 0], label="Pusher")  # 显示推杆随时间变化的位置
ax.plot(episodes[0][0][:, 1], label="Object")  # 显示物体随时间变化的位置
ax.set(xlabel="Step", ylabel="Position (m)", title="One recorded episode")  # 明确时间轴和物理单位
ax.legend()  # 标明两条曲线的含义
plt.show()  # 显示真实轨迹图


### 怎样读这张图

横轴是环境步数，不是训练轮数；纵轴是位置。推杆变化而物体线段水平，说明还没有接触，或推杆正在离开。进入接触后，物体才跟着向右移动，两者维持规定间距。图中还没有网络预测，不能据此判断模型是否学会。

状态点比动作多一个。动作编号 0 把状态 0 变成状态 1，而不是状态 1 变成状态 2。

## 2. 训练网络预测变化量

输入是推杆位置、物体位置和本周期动作三个数；输出是两个位置的变化量。归一化统计只用前 80 条训练轨迹计算，后 20 条只做测试。变化量除以 0.025 米，避免小位移在数值上过小。

In [ ]:
def make_pairs(items):  # 将若干完整轨迹整理为状态转移样本
    x = np.concatenate([np.column_stack((s[:-1], a)) for s, a in items])  # 拼接执行前状态与对应区间动作
    y = np.concatenate([np.diff(s, axis=0) for s, a in items])  # 由相邻状态计算真实变化量
    return x, y  # 返回输入与监督答案
train_x, train_y = make_pairs(episodes[:80])  # 只从训练轨迹构造训练样本
test_x, test_y = make_pairs(episodes[80:])  # 独立构造未见轨迹的测试样本
mean = train_x.mean(axis=0)  # 使用训练数据计算输入中心
scale = train_x.std(axis=0) + 1e-6  # 使用训练数据计算尺度并防止除零
x_train = torch.tensor((train_x - mean) / scale)  # 将归一化输入转成张量
y_train = torch.tensor(train_y / 0.025)  # 将变化量缩放到适合训练的量级
model = nn.Sequential(nn.Linear(3, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))  # 建立变化量预测网络
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)  # 设置更新网络权重的优化器
losses = []  # 保存实际计算出的训练误差
for step in range(700):  # 执行多次随机小批量参数更新
    ids = torch.randint(len(x_train), (256,))  # 从训练数据随机抽取当前批次
    prediction = model(x_train[ids])  # 预测两个位置的变化量
    loss = ((prediction - y_train[ids]) ** 2).mean()  # 比较预测变化与真实变化
    optimizer.zero_grad()  # 清除上一次更新留下的梯度
    loss.backward()  # 计算当前误差对参数的影响
    optimizer.step()  # 更新网络参数
    losses.append(loss.item())  # 保存本次训练得到的误差
fig, ax = plt.subplots(figsize=(7, 3))  # 创建训练曲线画布
ax.plot(losses)  # 绘制每次更新的真实误差
ax.set(xlabel="Parameter update", ylabel="Scaled MSE", yscale="log")  # 使用对数轴看清后期误差变化
plt.show()  # 显示训练过程


### 怎样读训练曲线

横轴是参数更新次数，与上一图的环境步数不同。纵轴是缩放后的均方误差，不能直接读成厘米。整体下降说明模型更能拟合训练样本；局部波动来自每次抽到的样本不同。

低训练损失不证明连续预测可靠。下面固定一段新动作，分别每步重新给真实状态、以及一直使用自己的预测，两者共用同一个网络。

In [ ]:
def predict_delta(inputs):  # 定义不再更新参数的变化量预测函数
    with torch.no_grad():  # 评测时关闭梯度记录
        values = torch.tensor((np.asarray(inputs, dtype=np.float32) - mean) / scale)  # 使用训练阶段固定的归一化量
        return model(values).numpy() * 0.025  # 把预测变化恢复成米
test_delta = predict_delta(test_x)  # 计算测试样本的单步变化预测
object_mae = np.abs(test_delta[:, 1] - test_y[:, 1]).mean()  # 计算物体位置单步平均绝对误差
baseline_mae = np.abs(test_y[:, 1]).mean()  # 计算物体永远不动的基线误差
print(f"测试单步物体误差：{object_mae * 1000:.2f} mm；不动基线：{baseline_mae * 1000:.2f} mm")  # 用毫米报告实际误差
actions = np.full(30, 0.012, dtype=np.float32)  # 规定一段连续向右的未见动作
truth = [np.array([0.15, 0.28], dtype=np.float32)]  # 设置尚未接触的真实起点
rollout = [truth[0].copy()]  # 连续预测只在这里接收一次真实状态
one_step = [truth[0].copy()]  # 单步对照从相同起点开始
for action in actions:  # 逐个处理相同的动作序列
    one_step.append(truth[-1] + predict_delta(np.r_[truth[-1], action]))  # 每一步重新使用真实输入
    truth.append(transition(truth[-1], action))  # 获得用于核对的真实下一状态
    rollout.append(rollout[-1] + predict_delta(np.r_[rollout[-1], action]))  # 将上一步预测继续作为输入
truth, rollout, one_step = map(np.asarray, (truth, rollout, one_step))  # 整理成便于画图的数组
fig, ax = plt.subplots(figsize=(7, 3.5))  # 创建三种物体位置的对照图
ax.plot(truth[:, 1], color="black", label="Recorded truth")  # 黑线表示教学环境的真实结果
ax.plot(one_step[:, 1], linestyle=":", label="One-step prediction")  # 点线表示每步用真实输入的预测
ax.plot(rollout[:, 1], linestyle="--", label="Recursive prediction")  # 虚线表示连续使用自身结果的预测
ax.set(xlabel="Step", ylabel="Object position (m)")  # 标明时间与物体位置
ax.legend()  # 标明三种计算方式
plt.show()  # 显示预测与真实结果的差别
assert np.isfinite(rollout).all()  # 检查连续预测没有产生无效数字
assert object_mae < baseline_mae  # 检查模型至少优于本实验的不动基线


### 怎样解释最后的结果

先看黑线何时从水平变成上升：这是接触时间。点线每次从真实状态出发，不会把错误位置继续送入模型。虚线没有这种纠正，接触时刻一旦偏移，后面可能持续偏离。

三条线接近不代表任意动作都可靠。延长预测步数、改变初始间隙，或输入超过训练范围的动作，都可能增加偏差。打印的误差来自全部测试轨迹，最后一图只是一条对照轨迹，不能替代完整测试。

练习：将动作改成 -0.012，物体应保持不动；再改成超过训练范围的 0.06，记录失败而不是调整图轴隐藏它。不要用这个一维教学模型控制真实机械臂。